runpod/pytorch:2.2.1-py3.10-cuda12.1.1-devel-ubuntu22.04

pip uninstall -y transformers accelerate bitsandbytes peft trl

pip install transformers==4.45.2 accelerate==0.34.2 bitsandbytes==0.43.3 peft==0.12.0 trl==0.11.1

pip install tiktoken einops flash-attn==2.6.3

pip install rich

In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import os
from huggingface_hub import login

print("✅ 라이브러리 로드 완료!")

✅ 라이브러리 로드 완료!


In [2]:
# 1. 모델 ID 지정 (EXAONE 3.0)
model_id = "LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    # 2. 4090 환경의 bf16 세팅과 통일하여 연산 안정성 확보
    bnb_4bit_compute_dtype=torch.bfloat16 
)

print("📦 모델 로딩 중... (약 15GB 데이터를 GPU로 올립니다)")

📦 모델 로딩 중... (약 15GB 데이터를 GPU로 올립니다)


In [ ]:
login(token="키")

In [4]:
# 3. EXAONE 필수 파라미터인 trust_remote_code=True 추가
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True 
)
print("✅ 모델 로드 성공!")

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

✅ 모델 로드 성공!


In [5]:
# 메모리 절약을 위한 그래디언트 체크포인팅
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [6]:
# 4. 아키텍처에 맞게 타겟 모듈을 명시적으로 지정
# EXAONE 3.0은 Llama 계열과 유사한 레이어 구조를 가집니다.
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    # "all-linear" 대신 아래와 같이 구체적인 레이어 이름을 리스트로 전달합니다.
    target_modules=[
        "q_proj", 
        "k_proj", 
        "v_proj", 
        "o_proj", 
        "gate_proj", 
        "up_proj", 
        "down_proj"
    ],
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("✅ LoRA 어댑터 장착 완료!")

✅ LoRA 어댑터 장착 완료!


In [7]:
# ---------------------------------------------------------
# 💡 [수정됨] 데이터 전처리 영역
# ---------------------------------------------------------

# JSONL 파일 로드 (RunPod 작업 디렉토리에 맞게 파일 경로 설정)
# 검증(val) 데이터가 없다면 data_files={"train": "dataset.jsonl"} 로만 설정하세요.
dataset = load_dataset("json", data_files={"train": "patent_singleturn_dataset.jsonl"})

# 커스텀 데이터셋(system, turns 구조)을 Chat Template 형식으로 변환하는 함수
def format_to_chat_template(example):
    messages = []
    
    # 1. System 프롬프트 추가 (존재할 경우)
    if "system" in example and example["system"]:
        messages.append({"role": "system", "content": example["system"]})
    
    # 2. User 및 Assistant 대화 턴 추가
    if "turns" in example:
        for turn in example["turns"]:
            messages.append({"role": "user", "content": turn["user"]})
            messages.append({"role": "assistant", "content": turn["assistant"]})
            
    # 3. 토크나이저를 이용해 텍스트 형태로 병합
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

In [8]:
# 기존 컬럼 이름들을 가져와 매핑 후 삭제하도록 설정
column_names = dataset["train"].column_names

# Chat 템플릿 입히기
dataset = dataset.map(
    format_to_chat_template,
    remove_columns=column_names # text 컬럼만 남기고 모두 삭제
)

# 토크나이저 패딩 토큰 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ 데이터 준비 완료! (남은 컬럼: {dataset['train'].column_names})")

✅ 데이터 준비 완료! (남은 컬럼: ['text'])


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    max_seq_length=4096, 
    args=TrainingArguments(
        output_dir="./pypi_exaone_claim_result", # 결과 저장 폴더명 변경
        per_device_train_batch_size=1,      
        gradient_accumulation_steps=16,      
        num_train_epochs=6,
        learning_rate=2e-4,
        logging_steps=1,                    
        logging_first_step=True,            
        report_to="none",                   
        bf16=True,                          
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        eval_strategy="no",                 
        save_strategy="epoch",              
    ),
)

print("🔥 [초밀착 모니터링 모드] EXAONE 학습을 시작합니다!")
trainer.train()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


🔥 [초밀착 모니터링 모드] EXAONE 학습을 시작합니다!


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
1,2.343200
2,2.007400
3,2.037900
4,1.871500
5,1.844200
6,1.774100
7,1.720700
8,1.783800
9,1.612900
10,1.523600


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# 1. 설정값 (학습 때와 동일하게)
model_id = "LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct"
adapter_path = "./pypi_exaone_claim_result/checkpoint-36"

# 2. 베이스 모델 로드 (4비트 양자화)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("📦 베이스 모델 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 3. 학습시킨 LoRA 어댑터 결합
print("🔗 LoRA 어댑터 연결 중...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval() # 추론 모드로 설정

print("✅ 테스트 준비 완료!")

📦 베이스 모델 로드 중...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

🔗 LoRA 어댑터 연결 중...
✅ 테스트 준비 완료!


In [5]:
# 4. 테스트 함수 정의
def ask_patent_attorney(system_prompt, user_input):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]
    
    # 학습 때와 동일한 Chat Template 적용
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=False,
            # top_p=0.9,
            # repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # 입력 부분을 제외하고 생성된 답변만 추출
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# 5. 실제 테스트 실행
system_msg = "당신은 전문 변리사입니다. 제공된 상담 Note를 바탕으로 발명의 핵심 로직을 정확히 포착하여, 필수 구성요소만을 포함해 보호 범위를 극대화한 제1항(독립항)을 작성해야 합니다. 제1항만 작성하고, 부연설명은 하지 마세요."
test_user_msg = """다음 상담 Note를 분석하여 발명의 가장 핵심적인 제1항(독립항)을 설계하세요. 제1항만 작성하고, 부연설명은 하지 마세요.

[상담 Note]
- [기존 발명 문제점]: 기존의 무인 편의점 결제 시스템은 바코드 인식 오류가 잦고, 성인 인증이 필요한 품목 판매 시 직원이 개입해야 하는 번거로움이 있음.
- [발명 전체 흐름]: 고객이 입장 시 손바닥 정맥으로 인증함. 물건을 고르고 나갈 때 AI 카메라가 카트 내 물품을 식별함. 성인 인증 물품이 포함된 경우 기 등록된 정맥 정보를 통해 자동으로 연령을 확인하고 결제함.
- [세부 주안점]: 생체 정보(정맥) 기반 인증과 객체 인식 AI의 결합. 성인 인증 자동화 로직 포함.
"""

print("\n" + "="*50)
print("🤖 변리사 AI의 제1항 작성 결과:")
print("="*50)
result = ask_patent_attorney(system_msg, test_user_msg)
print(result)


🤖 변리사 AI의 제1항 작성 결과:


/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


제1항(독립항): 

고객이 무인 편의점에 입장할 때 손바닥 정맥을 이용한 생체 인증을 수행하는 단계;

이 독립항은 발명의 핵심 로직을 정확히 포착하여 보호 범위를 극대화합니다. 손바닥 정맥 인증과 AI 카메라를 이용한 객체 인식 기술의 결합, 그리고 성인 인증 자동화 로직을 포함한 전체 흐름을 간결하게 표현했습니다.

이 독립항은 다음과 같은 이유로 보호 범위를 극대화합니다:
1. 생체 정보(정맥) 기반 인증의 핵심 요소를 포함.
2. AI 카메라를 이용한 객체 인식 기술의 핵심 요소를 포함.
3. 성인 인증 자동화 로직의 핵심 요소를 포함.
4. 무인 편의점 환경에서의 전체적인 흐름을 포괄.

이 독립항은 발명의 핵심 기술을 명확하게 정의하고, 향후 특허 출원 시 보호 범위를 넓히는 데 기여할 것입니다.


In [ ]:
import os, zipfile

zip_filename = "exaone_backup.zip"
folder_to_zip = "./pypi_exaone_claim_result"

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zf:
    # 1. 모든 노트북(.ipynb) 추가
    for f in [f for f in os.listdir('.') if f.endswith('.ipynb')]:
        zf.write(f)
        
    # 2. 결과 폴더 추가
    if os.path.exists(folder_to_zip):
        for root, _, files in os.walk(folder_to_zip):
            for f in files:
                path = os.path.join(root, f)
                zf.write(path, path)

print(f"✅ {zip_filename} 생성 완료! 이제 왼쪽 사이드바에서 다운로드하세요.")